# Drug Review Insights & Summarization Tool: API Integration & Prompt Engineering

**Phase 3 — Alex**

This notebook:
- Loads Nancy's cleaned outputs (`cleaned_drug_reviews.csv`, `drug_review_profile.csv`)
- Builds structured data profiles for three query types: **overall dataset**, **per-drug**, and **per-condition**
- Sends each profile to the OpenAI API and returns plain-language summaries
- Iterates on prompts to improve accuracy and relevance
- Saves all narrative outputs to `/data/final/ai_summaries.json` for Michael's visualization layer

Output file:
- `data/final/ai_summaries.json`

## Install & Import Packages

In [ ]:
# Install OpenAI SDK if running in Colab
# !pip install openai python-dotenv -q

In [ ]:
import os
import json
import textwrap
import pandas as pd
import numpy as np
from openai import OpenAI

# Load API key from .env file (never hardcode — never push to GitHub)
# In Colab, use: from google.colab import userdata
#                client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
# Locally with .env:
from dotenv import load_dotenv
load_dotenv()  # reads .env in the project root

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("OpenAI client ready." if os.getenv("OPENAI_API_KEY") else "⚠️  API key not found — check your .env or Colab secret.")

## Load Nancy's Cleaned Outputs

Nancy's preprocessing exports two files we consume here:
- `cleaned_drug_reviews.csv` — full cleaned dataset (219K rows) with features: `drugName`, `condition`, `review`, `rating`, `sentiment`, `review_length`, `weighted_rating`, `source`
- `drug_review_profile.csv` — high-level summary statistics

In [ ]:
# Load Nancy's output files
cleaned_df = pd.read_csv("../data/final/cleaned_drug_reviews.csv")
profile_df = pd.read_csv("../data/final/drug_review_profile.csv")

print("Cleaned dataset shape:", cleaned_df.shape)
print("\nSummary profile:")
print(profile_df.to_string(index=False))

## Build Structured Data Profiles for the AI

The AI cannot process 219K rows. Instead, we build **compact text summaries** that capture the key statistics and representative reviews for each query type:

1. **Overall dataset profile** — macro-level numbers + top drugs/conditions + sentiment breakdown
2. **Per-drug profile** — rating stats, sentiment split, and 3 sample reviews for a given drug
3. **Per-condition profile** — same structure, filtered by condition

These become the `{data_profile}` slot injected into each prompt.

In [ ]:
def build_overall_profile(cleaned_df: pd.DataFrame, profile_df: pd.DataFrame) -> str:
    """
    Build a compact overall dataset profile string for the AI prompt.
    Pulls macro stats, rating distribution, top drugs/conditions,
    and sentiment breakdown from Nancy's cleaned outputs.
    """
    # Summary statistics from profile_df
    stats = dict(zip(profile_df["metric"], profile_df["value"]))

    # Rating distribution
    rating_dist = (
        cleaned_df["rating"]
        .value_counts()
        .sort_index()
        .rename_axis("rating")
        .reset_index(name="count")
    )
    rating_str = ", ".join(
        f"rating {int(r)}: {int(c)} reviews"
        for r, c in zip(rating_dist["rating"], rating_dist["count"])
    )

    # Sentiment breakdown
    sentiment_counts = cleaned_df["sentiment"].value_counts()
    sentiment_str = ", ".join(
        f"{s}: {int(n)} ({100*n/len(cleaned_df):.1f}%)"
        for s, n in sentiment_counts.items()
    )

    # Top 10 drugs by review count
    top_drugs = (
        cleaned_df.groupby("drugName")
        .agg(review_count=("review", "count"), avg_rating=("rating", "mean"))
        .sort_values("review_count", ascending=False)
        .head(10)
        .reset_index()
    )
    drugs_str = "; ".join(
        f"{row.drugName} ({int(row.review_count)} reviews, avg rating {row.avg_rating:.1f})"
        for row in top_drugs.itertuples()
    )

    # Top 10 conditions by review count
    top_conditions = (
        cleaned_df.groupby("condition")
        .size()
        .sort_values(ascending=False)
        .head(10)
        .reset_index(name="count")
    )
    conditions_str = "; ".join(
        f"{row.condition} ({int(row['count'])} reviews)"
        for row in top_conditions.itertuples()
    )

    profile = f"""OVERALL DATASET PROFILE
=======================
Total reviews: {int(stats.get('total_rows', 0)):,}
Unique drugs: {int(stats.get('unique_drugs', 0)):,}
Unique conditions: {int(stats.get('unique_conditions', 0)):,}
Average rating: {float(stats.get('average_rating', 0)):.2f} / 10
Median rating: {float(stats.get('median_rating', 0)):.1f} / 10
Average review length: {float(stats.get('average_review_length', 0)):.0f} words

Rating distribution: {rating_str}

Sentiment breakdown: {sentiment_str}

Top 10 drugs by review volume: {drugs_str}

Top 10 conditions by review volume: {conditions_str}
"""
    return profile.strip()


# Preview
overall_profile = build_overall_profile(cleaned_df, profile_df)
print(overall_profile)

In [ ]:
def build_drug_profile(cleaned_df: pd.DataFrame, drug_name: str, n_samples: int = 3) -> str:
    """
    Build a per-drug profile string.
    Includes rating stats, sentiment split, and up to n_samples representative reviews.
    """
    drug_df = cleaned_df[cleaned_df["drugName"].str.lower() == drug_name.lower()]

    if drug_df.empty:
        return f"No reviews found for drug: {drug_name}"

    review_count = len(drug_df)
    avg_rating = drug_df["rating"].mean()
    median_rating = drug_df["rating"].median()
    sentiment_counts = drug_df["sentiment"].value_counts()

    # Sample reviews: one positive, one neutral, one negative (if they exist)
    samples = []
    for s in ["positive", "neutral", "negative"]:
        subset = drug_df[drug_df["sentiment"] == s]
        if not subset.empty:
            row = subset.sample(1).iloc[0]
            # Truncate long reviews
            review_text = str(row["review"])[:400]
            samples.append(f'[{s.upper()}, rating {int(row["rating"])}] "{review_text}..."')

    samples_str = "\n".join(samples) if samples else "No sample reviews available."

    # Top conditions this drug is used for
    top_conditions = (
        drug_df.groupby("condition")
        .size()
        .sort_values(ascending=False)
        .head(5)
        .index.tolist()
    )
    conditions_str = ", ".join(top_conditions)

    profile = f"""DRUG PROFILE: {drug_name.upper()}
{'='*(20+len(drug_name))}
Total reviews: {review_count:,}
Average rating: {avg_rating:.2f} / 10
Median rating: {median_rating:.1f} / 10
Positive reviews: {sentiment_counts.get('positive', 0)} ({100*sentiment_counts.get('positive',0)/review_count:.1f}%)
Neutral reviews:  {sentiment_counts.get('neutral', 0)} ({100*sentiment_counts.get('neutral',0)/review_count:.1f}%)
Negative reviews: {sentiment_counts.get('negative', 0)} ({100*sentiment_counts.get('negative',0)/review_count:.1f}%)

Top conditions treated: {conditions_str}

Sample patient reviews:
{samples_str}
"""
    return profile.strip()


# Preview with a common drug
drug_profile = build_drug_profile(cleaned_df, "Levothyroxine")
print(drug_profile)

In [ ]:
def build_condition_profile(cleaned_df: pd.DataFrame, condition: str, n_drugs: int = 5, n_samples: int = 3) -> str:
    """
    Build a per-condition profile string.
    Includes top drugs for the condition, rating stats, sentiment split,
    and sample patient reviews.
    """
    cond_df = cleaned_df[cleaned_df["condition"].str.lower() == condition.lower()]

    if cond_df.empty:
        return f"No reviews found for condition: {condition}"

    review_count = len(cond_df)
    avg_rating = cond_df["rating"].mean()
    median_rating = cond_df["rating"].median()
    sentiment_counts = cond_df["sentiment"].value_counts()

    # Top drugs for this condition
    top_drugs = (
        cond_df.groupby("drugName")
        .agg(review_count=("review", "count"), avg_rating=("rating", "mean"))
        .sort_values("review_count", ascending=False)
        .head(n_drugs)
        .reset_index()
    )
    drugs_str = "; ".join(
        f"{row.drugName} ({int(row.review_count)} reviews, avg {row.avg_rating:.1f})"
        for row in top_drugs.itertuples()
    )

    # Sample reviews
    samples = []
    for s in ["positive", "neutral", "negative"]:
        subset = cond_df[cond_df["sentiment"] == s]
        if not subset.empty:
            row = subset.sample(1).iloc[0]
            review_text = str(row["review"])[:400]
            samples.append(f'[{s.upper()}, {row["drugName"]}, rating {int(row["rating"])}] "{review_text}..."')

    samples_str = "\n".join(samples) if samples else "No sample reviews available."

    profile = f"""CONDITION PROFILE: {condition.upper()}
{'='*(22+len(condition))}
Total reviews: {review_count:,}
Average rating: {avg_rating:.2f} / 10
Median rating: {median_rating:.1f} / 10
Positive reviews: {sentiment_counts.get('positive', 0)} ({100*sentiment_counts.get('positive',0)/review_count:.1f}%)
Neutral reviews:  {sentiment_counts.get('neutral', 0)} ({100*sentiment_counts.get('neutral',0)/review_count:.1f}%)
Negative reviews: {sentiment_counts.get('negative', 0)} ({100*sentiment_counts.get('negative',0)/review_count:.1f}%)

Top drugs for this condition: {drugs_str}

Sample patient reviews:
{samples_str}
"""
    return profile.strip()


# Preview
condition_profile = build_condition_profile(cleaned_df, "Depression")
print(condition_profile)

## Prompt Engineering

We use three purpose-built prompts. The **system prompt** establishes the AI's role and output requirements. The **user prompt** injects the data profile via `{data_profile}`.

### Prompt Design Principles
- **Be specific about the audience** (pharmacist / researcher) so the AI calibrates vocabulary
- **Specify output structure** (numbered sections) so Michael can parse it programmatically
- **Constrain tone** (factual, no medical advice) to keep summaries safe and credible
- **Ask for explicit data citations** (averages, counts) to prevent hallucination

In [ ]:
# ── System prompt (shared across all query types) ────────────────────────────
SYSTEM_PROMPT = """You are a clinical data analyst assistant helping pharmacists and healthcare 
researchers interpret patient drug review data. Your role is to produce clear, factual, 
plain-language summaries of structured data profiles derived from patient-reported reviews.

Rules:
- Base every claim strictly on the numbers provided in the data profile. Do not infer or invent.
- Cite specific figures (averages, percentages, counts) to support each observation.
- Do not give medical advice or recommend specific treatments.
- Write in a professional but accessible tone appropriate for a pharmacist or researcher.
- Structure your response with clearly labeled sections as instructed.
- Keep each section concise (2–4 sentences).
"""

# ── User prompts (one per query type) ────────────────────────────────────────

OVERALL_PROMPT = """Below is a structured profile of a patient drug review dataset.

{data_profile}

Please produce a summary report with the following four sections:
1. Dataset Overview — size, scope, and key characteristics
2. Sentiment Landscape — overall positivity/negativity trends and what drives them
3. Notable Patterns — which drugs or conditions stand out and why
4. Limitations & Caveats — what a researcher should be cautious about when using this data
"""

DRUG_PROMPT = """Below is a structured profile of patient reviews for a specific medication.

{data_profile}

Please produce a medication summary report with the following four sections:
1. Patient Experience Overview — overall satisfaction level and rating patterns
2. Sentiment Breakdown — proportion of positive, neutral, and negative experiences and what the reviews reveal
3. Key Themes — recurring topics or concerns patients mention (infer only from the sample reviews provided)
4. Clinical Relevance — what a pharmacist should know when counseling patients on this medication
"""

CONDITION_PROMPT = """Below is a structured profile of patient reviews grouped by a specific medical condition.

{data_profile}

Please produce a condition summary report with the following four sections:
1. Condition Overview — patient volume, overall treatment satisfaction, and rating patterns
2. Drug Comparison — how top medications for this condition compare based on review counts and ratings
3. Patient Sentiment Themes — what positive and negative reviewers most commonly express (from sample reviews)
4. Research Implications — what gaps or patterns in this data are worth investigating further
"""

print("Prompts defined. System prompt length:", len(SYSTEM_PROMPT), "chars")
print("Overall prompt length:", len(OVERALL_PROMPT), "chars")
print("Drug prompt length:   ", len(DRUG_PROMPT), "chars")
print("Condition prompt length:", len(CONDITION_PROMPT), "chars")

## Core API Call Function

In [ ]:
def call_openai(system_prompt: str, user_prompt_template: str, data_profile: str,
                model: str = "gpt-4o", temperature: float = 0.3) -> str:
    """
    Send a data profile to OpenAI and return the narrative summary.

    Parameters
    ----------
    system_prompt : str
        The role/behavior instruction for the AI.
    user_prompt_template : str
        User-turn template containing a {data_profile} placeholder.
    data_profile : str
        The structured text profile built from Nancy's data.
    model : str
        OpenAI model to use. Default: gpt-4o.
    temperature : float
        Lower = more factual/deterministic. Recommended range: 0.1–0.4.

    Returns
    -------
    str
        The AI narrative response.
    """
    user_message = user_prompt_template.format(data_profile=data_profile)

    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
    )
    return response.choices[0].message.content.strip()

print("API call function ready.")

## Generate Overall Dataset Summary

In [ ]:
overall_profile = build_overall_profile(cleaned_df, profile_df)

overall_summary = call_openai(
    system_prompt=SYSTEM_PROMPT,
    user_prompt_template=OVERALL_PROMPT,
    data_profile=overall_profile,
)

print("═" * 60)
print("OVERALL DATASET SUMMARY")
print("═" * 60)
print(overall_summary)

## Generate Per-Drug Summaries (Top 5 Drugs by Review Volume)

In [ ]:
# Select top 5 drugs by review count for demonstration
top_5_drugs = (
    cleaned_df.groupby("drugName")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .index.tolist()
)

print("Generating summaries for:", top_5_drugs)

drug_summaries = {}

for drug in top_5_drugs:
    print(f"\n→ Calling API for: {drug}")
    profile = build_drug_profile(cleaned_df, drug)
    summary = call_openai(
        system_prompt=SYSTEM_PROMPT,
        user_prompt_template=DRUG_PROMPT,
        data_profile=profile,
    )
    drug_summaries[drug] = {
        "data_profile": profile,
        "summary": summary
    }
    print(f"  ✓ Done ({len(summary)} chars)")

print("\nAll drug summaries generated.")

In [ ]:
# Inspect one example
example_drug = top_5_drugs[0]
print("═" * 60)
print(f"DRUG SUMMARY: {example_drug.upper()}")
print("═" * 60)
print(drug_summaries[example_drug]["summary"])

## Generate Per-Condition Summaries (Top 5 Conditions by Review Volume)

In [ ]:
# Select top 5 conditions by review count (excluding Unknown)
top_5_conditions = (
    cleaned_df[cleaned_df["condition"] != "Unknown"]
    .groupby("condition")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .index.tolist()
)

print("Generating summaries for:", top_5_conditions)

condition_summaries = {}

for condition in top_5_conditions:
    print(f"\n→ Calling API for: {condition}")
    profile = build_condition_profile(cleaned_df, condition)
    summary = call_openai(
        system_prompt=SYSTEM_PROMPT,
        user_prompt_template=CONDITION_PROMPT,
        data_profile=profile,
    )
    condition_summaries[condition] = {
        "data_profile": profile,
        "summary": summary
    }
    print(f"  ✓ Done ({len(summary)} chars)")

print("\nAll condition summaries generated.")

In [ ]:
# Inspect one example
example_condition = top_5_conditions[0]
print("═" * 60)
print(f"CONDITION SUMMARY: {example_condition.upper()}")
print("═" * 60)
print(condition_summaries[example_condition]["summary"])

## Prompt Iteration Log

Below are the prompt versions tested and the key improvements made. This documents the iterative engineering process required by Phase 3.

| Version | Change Made | Problem Solved |
|---------|-------------|----------------|
| v1 | No system prompt, open-ended user prompt | AI gave generic medical advice not grounded in the data |
| v2 | Added system prompt with "base claims on data provided" rule | Reduced hallucination; AI still added unsolicited recommendations |
| v3 | Added "do not give medical advice" rule + numbered output sections | Eliminated recommendations; structured output parseable by Michael |
| v4 (current) | Added `temperature=0.3` + explicit cite-figures instruction | Most consistent, factual outputs across repeated calls |

## Interactive Query Function

This utility lets Mark's `main.py` (Phase 5) query any drug or condition on demand.

In [ ]:
def generate_summary(query_type: str, query_value: str = None) -> str:
    """
    Public-facing function called by main.py.

    Parameters
    ----------
    query_type : str
        One of 'overall', 'drug', or 'condition'.
    query_value : str, optional
        Required when query_type is 'drug' or 'condition'.

    Returns
    -------
    str
        Plain-language AI narrative summary.
    """
    if query_type == "overall":
        profile = build_overall_profile(cleaned_df, profile_df)
        return call_openai(SYSTEM_PROMPT, OVERALL_PROMPT, profile)

    elif query_type == "drug":
        if not query_value:
            raise ValueError("query_value (drug name) is required for query_type='drug'")
        profile = build_drug_profile(cleaned_df, query_value)
        return call_openai(SYSTEM_PROMPT, DRUG_PROMPT, profile)

    elif query_type == "condition":
        if not query_value:
            raise ValueError("query_value (condition name) is required for query_type='condition'")
        profile = build_condition_profile(cleaned_df, query_value)
        return call_openai(SYSTEM_PROMPT, CONDITION_PROMPT, profile)

    else:
        raise ValueError(f"Unknown query_type '{query_type}'. Use 'overall', 'drug', or 'condition'.")


# Quick smoke test (comment out after first successful run to save API credits)
# print(generate_summary("drug", "Metformin"))

## Save All Summaries to `ai_summaries.json`

This file is the handoff to Michael (Phase 4) and Mark's `main.py` (Phase 5).

In [ ]:
output = {
    "overall": {
        "data_profile": overall_profile,
        "summary": overall_summary
    },
    "drugs": drug_summaries,
    "conditions": condition_summaries
}

os.makedirs("../data/final", exist_ok=True)
output_path = "../data/final/ai_summaries.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"✓ Saved AI summaries to {output_path}")
print(f"  Keys: {list(output.keys())}")
print(f"  Drug summaries: {len(drug_summaries)}")
print(f"  Condition summaries: {len(condition_summaries)}")

## Verify Output Structure

Confirm the JSON is readable and structured correctly before pushing to GitHub.

In [ ]:
with open("../data/final/ai_summaries.json", "r") as f:
    verification = json.load(f)

print("Top-level keys:", list(verification.keys()))
print("\nOverall summary preview (first 300 chars):")
print(verification["overall"]["summary"][:300], "...")

print("\nDrug summary keys:", list(verification["drugs"].keys()))
print("Condition summary keys:", list(verification["conditions"].keys()))
print("\n✓ Output file verified and ready for Phase 4 (Michael) and Phase 5 (Mark).")